# 1. Importación de Librerías
Empezamos importando todas las librerías que necesitaremos para este notebook:
- `scanpy` para el manejo de datos de single-cell.
- `pandas` y `numpy` para la manipulación de datos.
- `sys` para añadir nuestra carpeta de código fuente al path.
- Función personalizada `fetch_gene_symbols_ensembl`.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
import os

Añadimos la carpeta 'src' al path para poder importar nuestros módulos. El notebook está en la carpeta 'notebooks' y 'src' está al mismo nivel que 'notebooks'.
Importamos las funciones propias.

In [ ]:
sys.path.append('../src')
from ensembl_api import fetch_gene_symbols_ensembl
import preprocessing as prep

# 2. Carga del Dataset Crudo
Definimos las rutas a nuestros datos usando rutas relativas.
Luego, cargamos el dataset original de scRNA-seq en un objeto `AnnData`.

Definimos las rutas relativas

In [ ]:
DATA_RAW_PATH = '../data/raw/'
DATA_PROCESSED_PATH = '../data/processed/'
RAW_FILENAME = '76c942bd-45c3-47ec-b290-1e695ec9c177.h5ad'

Cargamos los datos

In [ ]:
adata_raw = sc.read_h5ad(os.path.join(DATA_RAW_PATH, RAW_FILENAME), backed='r')

Mostramos un resumen del objeto para verificar la carga

In [ ]:
print("Resumen del dataset original:")
print(adata_raw)

Se analiza la distribución de células por enfermedad y por tipo celular para decidir cómo enfocar nuestro estudio.

Recuento de células por enfermedad

In [ ]:
print("\nRecuento de células por enfermedad:")
print(adata_raw.obs['disease'].value_counts()) #Poner adata_processed???

Recuento de células por tipo celular (total)

In [ ]:
print("\nRecuento de células por tipo celular (total):")
print(adata_raw.obs['cell_type'].value_counts())

Tabla cruzada para ver la distribución de tipos celulares por enfermedad


In [ ]:
print("\nTabla cruzada de Enfermedad vs. Tipo Celular:")
display(pd.crosstab(adata_raw.obs['disease'], adata_raw.obs['cell_type']))


# 3. Selección del Contexto: Cáncer de Pulmón

Basado en el análisis, seleccionamos las células de 'lung cancer' como nuestro dataset de interés. Realizamos este filtrado ANTES de cualquier pre-procesamiento intensivo para optimizar el uso de recursos y enfocar el análisis.

In [ ]:
print("\n--- Filtrando por Cáncer de Pulmón ---")

# Filtramos el objeto en modo 'backed' para crear una vista
adata_lung_view = adata_raw[adata_raw.obs['disease'] == 'lung cancer']

# Usamos .to_memory() para cargar esa vista en un nuevo objeto en RAM
adata_lung_in_memory = adata_lung_view.to_memory()

# Liberamos la memoria del objeto grande en disco
del adata_raw
import gc; gc.collect()

print(f"Dataset de Cáncer de Pulmón con {adata_lung_in_memory.n_obs} células.")

# 4. Pipeline de Verificación y Pre-procesamiento
Se realiza una serie de comprobaciones para aplicar cada paso del pre-procesamiento de forma condicional, llamando a funciones específicas desde nuestro módulo `src/preprocessing.py`.

In [ ]:
print("\n--- Iniciando pipeline de verificación y pre-procesamiento ---")

In [ ]:
adata_processed = adata_lung_in_memory.copy()
del adata_lung_in_memory; import gc; gc.collect()

Definimos los parámetros que queremos para el pre-procesamiento

In [ ]:
preprocessing_params = {
    'qc_metrics':   {'min_genes': 200, 'min_cells': 3, 'mt_qc_threshold': 15},
    'norm_log':     {'target_sum': 1e4},
    'hvg':          {'n_top_genes': 3000, 'flavor': 'seurat_v3'},
    'pca_umap':     {'scale_max_value':10,'pca_svd_solver': 'arpack','n_pcs':30}    
}

Ejecutamos la función maestra

In [ ]:
state = prep.validate_adata_state(adata_processed)

In [ ]:
if not state['qc_metrics_calculated']:
    adata_processed = prep.calculate_and_filter_qc(adata_processed, **preprocessing_params['qc_metrics'])


In [ ]:
if not state['is_normalized_and_logged']:
    adata_processed = prep.normalize_and_log_transform(adata_processed, **preprocessing_params['norm_log'])

#TODO: Añadir print de estado

In [ ]:
if not state['hvg_calculated']:
    adata_processed = prep.find_highly_variable_genes(adata_processed, **preprocessing_params['hvg'] )

In [ ]:
print("Pre-procesamiento base (QC, Norm, Log, HVG-marking) completado.")
print("La matriz .X sigue siendo dispersa:", type(adata_processed.X))

## 4.2  Análisis de Reducción de Dimensionalidad para Visualización

In [ ]:
print("\n--- Iniciando análisis para visualización ---")
adata_processed = prep.annotate_for_visualization(adata_processed, **preprocessing_params['pca_umap'])

Visualizamos la calidad final de los datos

In [ ]:
print("\nGenerando UMAP de los tipos celulares finales (después de filtrar)...")
sc.pl.umap(adata_processed, color='cell_type', title='UMAP de Tipos Celulares')

In [ ]:
sc.pl.violin(adata_processed, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, show=True)


# 5. Limpieza de tipos celulares raros
Para asegurar que nuestro modelo de Machine Learning tenga suficientes ejemplos de cada clase para aprender, vamos a eliminar los tipos celulares que son muy raros en nuestro dataset. Establecemos un umbral mínimo de células por cada tipo.

Calculamos el recuento de cada tipo celular en el dataset de pulmón

In [ ]:
cell_counts_lung = adata_processed.obs['cell_type'].value_counts()
print("Recuento de tipos celulares antes del filtrado:")
print(cell_counts_lung)

Definimos un umbral mínimo

In [ ]:
min_cell_count_threshold = 100

Obtenemos la lista de tipos celulares que cumplen con el umbral

In [ ]:
tipos_a_mantener = cell_counts_lung[cell_counts_lung >= min_cell_count_threshold].index

Filtramos el objeto AnnData final

In [ ]:
adata_final = adata_processed[adata_processed.obs['cell_type'].isin(tipos_a_mantener)].copy()
print(f"\nSe han mantenido los tipos celulares con >= {min_cell_count_threshold} células.")
print(f"El dataset final para el modelado contiene {adata_final.n_obs} células.")
print("\nRecuento de tipos celulares final:")
print(adata_final.obs['cell_type'].value_counts())

In [ ]:
sc.pl.umap(adata_final, color='cell_type', title='UMAP de Tipos Celulares en Cáncer de Pulmón')

# 6. Anotación de Nombres de Genes

 Los identificadores de genes en nuestro dataset están en formato Ensembl ID (ej. `ENSG...`), que no es fácilmente interpretable. Para facilitar el análisis biológico, utilizaremos nuestra función personalizada que llama a la API de Ensembl para obtener los símbolos de genes correspondientes (ej. `CD3D`).


In [ ]:
print("\n--- Anotando Nombres de Genes ---")

Obtenemos la lista de Ensembl IDs del índice de .var

In [ ]:
ensembl_ids = adata_final.var.index.tolist()

Llamamos a nuestra función importada desde src/ensembl_api.py

In [ ]:
print(f"Obteniendo símbolos para {len(ensembl_ids)} genes desde la API de Ensembl... (esto puede tardar)")
gene_symbol_dict = fetch_gene_symbols_ensembl(ensembl_ids)

Añadimos la nueva columna 'gene_name' a los metadatos de los genes (.var)

In [ ]:
adata_final.var['gene_name'] = adata_final.var.index.map(gene_symbol_dict)

Para los IDs que no devolvieron un símbolo, usamos el 'feature_name' como respaldo

In [ ]:
adata_final.var['gene_name'] = adata_final.var['gene_name'].fillna(adata_final.var['feature_name'])

print("Anotación de genes completada.")
print(f"Número de genes sin nombre de símbolo encontrado (NaNs rellenados): {adata_final.var['gene_name'].isna().sum()}")
print("\nEjemplo de la anotación de genes:")
display(adata_final.var[['feature_name', 'gene_name']].head())

# 7. Guardado del Dataset Final

Guardamos el objeto `AnnData` limpio y anotado en un nuevo fichero `.h5ad` en la carpeta `data/processed/`. Se utilizará posteriormente en el siguiente notebook para entrenar los modelos de Machine Learning.

In [ ]:
print("\n--- Guardando el Dataset Procesado ---")

PROCESSED_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
final_path = os.path.join(DATA_PROCESSED_PATH, PROCESSED_FILENAME)

adata_final.write(final_path)

print(f"Dataset procesado guardado exitosamente en: {final_path}")